# GNN Retrieval Training from Neo4j (`Chunk`-`Entity`)

Notebook строит обучающий пайплайн для retrieval на графе Neo4j:
- читает структуру `(:Chunk)-[:MENTIONS]->(:Entity)` и `(:Entity)-[:RELATED_TO]->(:Entity)`,
- формирует гетерогенный граф,
- обучает GraphSAGE для ранжирования `Entity` по `Chunk` (link prediction),
- считает `Recall@K`/`MRR`,
- записывает `gnn_embedding` обратно в Neo4j.


In [ ]:
# При необходимости раскомментируйте установку зависимостей
# %pip install neo4j pandas numpy scikit-learn torch torch-geometric qdrant-client


In [8]:
import os
import random

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from neo4j import GraphDatabase
from qdrant_client import QdrantClient
from qdrant_client.models import FieldCondition, Filter, MatchAny
from torch import nn
from torch_geometric.data import HeteroData
from torch_geometric.nn import HeteroConv, SAGEConv

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DEVICE


device(type='cuda')

In [10]:
# Конфиг подключения
NEO4J_URI = os.getenv('NEO4J_URI', 'bolt://localhost:7687')
NEO4J_USER = os.getenv('NEO4J_USER', 'neo4j')
NEO4J_PASSWORD = os.getenv('NEO4J_PASSWORD', 'password')

QDRANT_URL = os.getenv('QDRANT_URL', 'http://localhost:6333')
QDRANT_API_KEY = os.getenv('QDRANT_API_KEY')
QDRANT_CHUNK_COLLECTION = os.getenv('QDRANT_CHUNK_COLLECTION', 'legal_chunks_vectors')
QDRANT_ENTITY_COLLECTION = os.getenv('QDRANT_ENTITY_COLLECTION', 'legal_entities_vectors')
QDRANT_ID_FIELD = os.getenv('QDRANT_ID_FIELD', 'id')

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
qdrant = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY)

def run_query(query: str, params: dict | None = None):
    with driver.session() as session:
        result = session.run(query, params or {})
        return [r.data() for r in result]

schema_probe = run_query("""
MATCH (c:Chunk)
OPTIONAL MATCH (c)-[:MENTIONS]->(e:Entity)
RETURN count(DISTINCT c) AS chunks, count(DISTINCT e) AS entities, count(*) AS mention_rows
""")
schema_probe


[{'chunks': 600, 'entities': 2172, 'mention_rows': 3492}]

In [11]:
# Вытягиваем узлы/ребра и embedding_id для загрузки векторов из Qdrant
chunks = run_query("""
MATCH (c:Chunk)
RETURN
  c.chunk_id AS chunk_id,
  c.doc_id AS doc_id,
  c.embedding_id AS embedding_id,
  coalesce(c.text, '') AS text
""")

entities = run_query("""
MATCH (e:Entity)
RETURN
  e.entity_id AS entity_id,
  coalesce(e.embedding_id, e.entity_id) AS embedding_id,
  coalesce(e.entity_name, e.normalized_value, e.value, '') AS entity_text
""")

mentions = run_query("""
MATCH (c:Chunk)-[:MENTIONS]->(e:Entity)
RETURN c.chunk_id AS chunk_id, e.entity_id AS entity_id
""")

related = run_query("""
MATCH (e1:Entity)-[:RELATED_TO]->(e2:Entity)
RETURN e1.entity_id AS src_entity_id, e2.entity_id AS dst_entity_id
""")

chunks_df = pd.DataFrame(chunks).dropna(subset=['chunk_id', 'embedding_id']).drop_duplicates('chunk_id')
entities_df = pd.DataFrame(entities).dropna(subset=['entity_id', 'embedding_id']).drop_duplicates('entity_id')
mentions_df = pd.DataFrame(mentions).dropna().drop_duplicates()
related_df = pd.DataFrame(related).dropna().drop_duplicates()

print('chunks (with embedding_id):', len(chunks_df))
print('entities (with embedding_id):', len(entities_df))
print('mentions:', len(mentions_df))
print('related_to:', len(related_df))
chunks_df.head(2), entities_df.head(2), mentions_df.head(2)


Received notification from DBMS server: <GqlStatusObject gql_status='01N52', status_description='warn: property key does not exist. The property `text` does not exist. Verify that the spelling is correct.', position=<SummaryInputPosition line=7, column=14, offset=119>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 119, 'line': 7, 'column': 14}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "\nMATCH (c:Chunk)\nRETURN\n  c.chunk_id AS chunk_id,\n  c.doc_id AS doc_id,\n  c.embedding_id AS embedding_id,\n  coalesce(c.text, '') AS text\n"
Received notification from DBMS server: <GqlStatusObject gql_status='01N52', status_description='warn: property key does not exist. The property `normalized_value` does not exist. Verify tha

chunks (with embedding_id): 600
entities (with embedding_id): 2172
mentions: 3492
related_to: 2285


(                               chunk_id doc_id  \
 0  019d8288-d132-71ed-b343-9c65524dcb53     12   
 1  019d8288-d136-7bfd-accb-06a906af2fe0     12   
 
                            embedding_id text  
 0  b22e00fa-af96-5b15-8d49-4f770d79cc21       
 1  d02ca84d-1bca-5a0f-8ec0-08810d645b90       ,
                                            entity_id  \
 0  f7381fc918f9933102106f5799248780039f5987967067...   
 1  8f4e8b62535fb495bb91ff066c856652416cdd676a8615...   
 
                            embedding_id        entity_text  
 0  68363780-c85c-5712-be9c-d811cab98174      Paris Commune  
 1  124ebd02-c765-5073-82f5-174cc956e4ed  Russian Civil War  ,
                                chunk_id  \
 0  019d8288-d132-71ed-b343-9c65524dcb53   
 1  019d8288-d132-71ed-b343-9c65524dcb53   
 
                                            entity_id  
 0  2246cc1e9afc11f58ec447ae8eb7cd7f6091f5b4672428...  
 1  15b38175c1f72394880594a9d93b640e5d14c9db0029ae...  )

In [12]:
def _normalize_qdrant_point_id(raw_id):
    if raw_id is None or pd.isna(raw_id):
        return None
    if isinstance(raw_id, (int, np.integer)):
        return int(raw_id)

    raw_id = str(raw_id).strip()
    if not raw_id:
        return None
    if raw_id.lstrip('-').isdigit():
        return int(raw_id)
    return raw_id

# Загружаем векторы из Qdrant по embedding_id
def _extract_vector(raw_vector):
    if raw_vector is None:
        return None
    if isinstance(raw_vector, dict):
        # named vectors: берём первый
        raw_vector = next(iter(raw_vector.values())) if raw_vector else None
    if raw_vector is None:
        return None
    return np.asarray(raw_vector, dtype=np.float32)

def fetch_vectors_by_embedding_ids(collection_name: str, embedding_ids: list[str], batch_size: int = 256):
    vectors: dict[str, np.ndarray] = {}
    ids = []
    for raw_id in embedding_ids:
        normalized_id = _normalize_qdrant_point_id(raw_id)
        if normalized_id is not None:
            ids.append(normalized_id)

    ids = list(dict.fromkeys(ids))
    for i in range(0, len(ids), batch_size):
        batch = ids[i:i+batch_size]
        points = qdrant.retrieve(
            collection_name=collection_name,
            ids=batch,
            with_payload=False,
            with_vectors=True,
        )
        for p in points:
            v = _extract_vector(p.vector)
            if v is not None:
                vectors[str(p.id)] = v
    return vectors

chunk_vec_map = fetch_vectors_by_embedding_ids(
    collection_name=QDRANT_CHUNK_COLLECTION,
    embedding_ids=chunks_df['embedding_id'].astype(str).tolist(),
)
entity_vec_map = fetch_vectors_by_embedding_ids(
    collection_name=QDRANT_ENTITY_COLLECTION,
    embedding_ids=entities_df['embedding_id'].astype(str).tolist(),
)

chunks_df['vec'] = chunks_df['embedding_id'].astype(str).map(chunk_vec_map)
entities_df['vec'] = entities_df['embedding_id'].astype(str).map(entity_vec_map)

chunks_df = chunks_df[chunks_df['vec'].notna()].copy().reset_index(drop=True)
entities_df = entities_df[entities_df['vec'].notna()].copy().reset_index(drop=True)

if chunks_df.empty or entities_df.empty:
    raise ValueError('Не удалось получить вектора из Qdrant. Проверьте collection names и embedding_id mapping.')

chunk_dims = chunks_df['vec'].apply(len).unique().tolist()
entity_dims = entities_df['vec'].apply(len).unique().tolist()
if len(chunk_dims) != 1 or len(entity_dims) != 1:
    raise ValueError(f'Непостоянная размерность векторов: chunk={chunk_dims}, entity={entity_dims}')

chunk_x = np.vstack(chunks_df['vec'].values).astype(np.float32)
entity_x = np.vstack(entities_df['vec'].values).astype(np.float32)

# L2-нормализация входных фичей
chunk_x = chunk_x / (np.linalg.norm(chunk_x, axis=1, keepdims=True) + 1e-12)
entity_x = entity_x / (np.linalg.norm(entity_x, axis=1, keepdims=True) + 1e-12)

chunk2idx = {cid: i for i, cid in enumerate(chunks_df['chunk_id'].tolist())}
entity2idx = {eid: i for i, eid in enumerate(entities_df['entity_id'].tolist())}

mentions_df = mentions_df[mentions_df['chunk_id'].isin(chunk2idx) & mentions_df['entity_id'].isin(entity2idx)].copy()
related_df = related_df[related_df['src_entity_id'].isin(entity2idx) & related_df['dst_entity_id'].isin(entity2idx)].copy()

mention_src = torch.tensor([chunk2idx[x] for x in mentions_df['chunk_id']], dtype=torch.long)
mention_dst = torch.tensor([entity2idx[x] for x in mentions_df['entity_id']], dtype=torch.long)

rel_src = torch.tensor([entity2idx[x] for x in related_df['src_entity_id']], dtype=torch.long) if len(related_df) else torch.empty(0, dtype=torch.long)
rel_dst = torch.tensor([entity2idx[x] for x in related_df['dst_entity_id']], dtype=torch.long) if len(related_df) else torch.empty(0, dtype=torch.long)

data = HeteroData()
data['chunk'].x = torch.from_numpy(chunk_x)
data['entity'].x = torch.from_numpy(entity_x)

data['chunk', 'mentions', 'entity'].edge_index = torch.stack([mention_src, mention_dst], dim=0)
data['entity', 'mentioned_in', 'chunk'].edge_index = torch.stack([mention_dst, mention_src], dim=0)
data['entity', 'related_to', 'entity'].edge_index = torch.stack([rel_src, rel_dst], dim=0)
data['entity', 'related_to_rev', 'entity'].edge_index = torch.stack([rel_dst, rel_src], dim=0)

print(f'Loaded Qdrant vectors: chunks={len(chunks_df)} (dim={chunk_x.shape[1]}), entities={len(entities_df)} (dim={entity_x.shape[1]})')
data


Loaded Qdrant vectors: chunks=600 (dim=256), entities=2172 (dim=256)


HeteroData(
  chunk={ x=[600, 256] },
  entity={ x=[2172, 256] },
  (chunk, mentions, entity)={ edge_index=[2, 3492] },
  (entity, mentioned_in, chunk)={ edge_index=[2, 3492] },
  (entity, related_to, entity)={ edge_index=[2, 2285] },
  (entity, related_to_rev, entity)={ edge_index=[2, 2285] }
)

In [13]:
# Train/Val/Test split по рёбрам MENTIONS
all_edges = data['chunk', 'mentions', 'entity'].edge_index.t().cpu().numpy()
perm = np.random.permutation(len(all_edges))
all_edges = all_edges[perm]

n = len(all_edges)
n_train = int(0.8 * n)
n_val = int(0.1 * n)

train_edges = torch.tensor(all_edges[:n_train], dtype=torch.long)
val_edges = torch.tensor(all_edges[n_train:n_train+n_val], dtype=torch.long)
test_edges = torch.tensor(all_edges[n_train+n_val:], dtype=torch.long)

num_chunks = data['chunk'].x.shape[0]
num_entities = data['entity'].x.shape[0]

existing = set((int(c), int(e)) for c, e in all_edges.tolist())

def sample_negative(batch_edges: torch.Tensor, max_tries: int = 20) -> torch.Tensor:
    neg = batch_edges.clone()
    for i in range(len(neg)):
        c = int(neg[i, 0])
        for _ in range(max_tries):
            e = random.randrange(num_entities)
            if (c, e) not in existing:
                neg[i, 1] = e
                break
    return neg

len(train_edges), len(val_edges), len(test_edges)


(2793, 349, 350)

In [14]:
class RetrievalHeteroSAGE(nn.Module):
    def __init__(self, hidden_dim=256, out_dim=128):
        super().__init__()
        self.conv1 = HeteroConv({
            ('chunk', 'mentions', 'entity'): SAGEConv((-1, -1), hidden_dim),
            ('entity', 'mentioned_in', 'chunk'): SAGEConv((-1, -1), hidden_dim),
            ('entity', 'related_to', 'entity'): SAGEConv((-1, -1), hidden_dim),
            ('entity', 'related_to_rev', 'entity'): SAGEConv((-1, -1), hidden_dim),
        }, aggr='mean')
        self.conv2 = HeteroConv({
            ('chunk', 'mentions', 'entity'): SAGEConv((-1, -1), out_dim),
            ('entity', 'mentioned_in', 'chunk'): SAGEConv((-1, -1), out_dim),
            ('entity', 'related_to', 'entity'): SAGEConv((-1, -1), out_dim),
            ('entity', 'related_to_rev', 'entity'): SAGEConv((-1, -1), out_dim),
        }, aggr='mean')

    def forward(self, x_dict, edge_index_dict):
        x_dict = self.conv1(x_dict, edge_index_dict)
        x_dict = {k: F.relu(v) for k, v in x_dict.items()}
        x_dict = self.conv2(x_dict, edge_index_dict)
        x_dict = {k: F.normalize(v, p=2, dim=-1) for k, v in x_dict.items()}
        return x_dict

def edge_scores(chunk_emb: torch.Tensor, entity_emb: torch.Tensor, edges: torch.Tensor) -> torch.Tensor:
    c = chunk_emb[edges[:, 0]]
    e = entity_emb[edges[:, 1]]
    return (c * e).sum(dim=-1)

@torch.no_grad()
def evaluate_retrieval(model: nn.Module, data: HeteroData, eval_edges: torch.Tensor, k_list=(5, 10, 20)):
    model.eval()
    z = model(data.x_dict, data.edge_index_dict)
    chunk_z = z['chunk']
    entity_z = z['entity']

    recalls = {k: [] for k in k_list}
    mrr = []

    for c_idx, e_idx in eval_edges.tolist():
        scores = torch.matmul(entity_z, chunk_z[c_idx])
        ranking = torch.argsort(scores, descending=True)
        rank_pos = (ranking == e_idx).nonzero(as_tuple=False)
        if len(rank_pos) == 0:
            continue
        rank = int(rank_pos[0].item()) + 1

        for k in k_list:
            recalls[k].append(1.0 if rank <= k else 0.0)
        mrr.append(1.0 / rank)

    metrics = {f'Recall@{k}': float(np.mean(v)) if v else 0.0 for k, v in recalls.items()}
    metrics['MRR'] = float(np.mean(mrr)) if mrr else 0.0
    return metrics


In [15]:
model = RetrievalHeteroSAGE(hidden_dim=256, out_dim=128).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)

data = data.to(DEVICE)
train_edges = train_edges.to(DEVICE)
val_edges = val_edges.to(DEVICE)
test_edges = test_edges.to(DEVICE)

epochs = 30
for epoch in range(1, epochs + 1):
    model.train()
    optimizer.zero_grad()

    z = model(data.x_dict, data.edge_index_dict)
    pos_scores = edge_scores(z['chunk'], z['entity'], train_edges)
    neg_edges = sample_negative(train_edges.cpu()).to(DEVICE)
    neg_scores = edge_scores(z['chunk'], z['entity'], neg_edges)

    logits = torch.cat([pos_scores, neg_scores], dim=0)
    labels = torch.cat([torch.ones_like(pos_scores), torch.zeros_like(neg_scores)], dim=0)
    loss = F.binary_cross_entropy_with_logits(logits, labels)

    loss.backward()
    optimizer.step()

    if epoch % 5 == 0 or epoch == 1:
        val_metrics = evaluate_retrieval(model, data, val_edges)
        print(f'Epoch {epoch:03d} | loss={loss.item():.4f} | val={val_metrics}')

test_metrics = evaluate_retrieval(model, data, test_edges)
print('Test metrics:', test_metrics)


Epoch 001 | loss=0.6944 | val={'Recall@5': 0.0057306590257879654, 'Recall@10': 0.0057306590257879654, 'Recall@20': 0.014326647564469915, 'MRR': 0.00599371226356899}
Epoch 005 | loss=0.6708 | val={'Recall@5': 0.06303724928366762, 'Recall@10': 0.0830945558739255, 'Recall@20': 0.1318051575931232, 'MRR': 0.044032141397959336}
Epoch 010 | loss=0.5841 | val={'Recall@5': 0.14040114613180515, 'Recall@10': 0.17478510028653296, 'Recall@20': 0.2177650429799427, 'MRR': 0.09745767272502144}
Epoch 015 | loss=0.5457 | val={'Recall@5': 0.09742120343839542, 'Recall@10': 0.15472779369627507, 'Recall@20': 0.2607449856733524, 'MRR': 0.07371566560127679}
Epoch 020 | loss=0.5358 | val={'Recall@5': 0.13753581661891118, 'Recall@10': 0.19197707736389685, 'Recall@20': 0.2693409742120344, 'MRR': 0.09516971336293506}
Epoch 025 | loss=0.5330 | val={'Recall@5': 0.14040114613180515, 'Recall@10': 0.21203438395415472, 'Recall@20': 0.27507163323782235, 'MRR': 0.11588107996299683}
Epoch 030 | loss=0.5221 | val={'Recall@

In [ ]:
# Инференс эмбеддингов и запись обратно в Neo4j
model.eval()
with torch.no_grad():
    z = model(data.x_dict, data.edge_index_dict)

chunk_emb = z['chunk'].detach().cpu().numpy()
entity_emb = z['entity'].detach().cpu().numpy()

chunk_records = [
    {'chunk_id': cid, 'gnn_embedding': emb.tolist()}
    for cid, emb in zip(chunks_df['chunk_id'].tolist(), chunk_emb)
]
entity_records = [
    {'entity_id': eid, 'gnn_embedding': emb.tolist()}
    for eid, emb in zip(entities_df['entity_id'].tolist(), entity_emb)
]

with driver.session() as session:
    session.run("""
    UNWIND $rows AS row
    MATCH (c:Chunk {chunk_id: row.chunk_id})
    SET c.gnn_embedding = row.gnn_embedding,
        c.gnn_model_version = $model_version,
        c.gnn_updated_at = datetime()
    """, rows=chunk_records, model_version='graphsage_v1')

    session.run("""
    UNWIND $rows AS row
    MATCH (e:Entity {entity_id: row.entity_id})
    SET e.gnn_embedding = row.gnn_embedding,
        e.gnn_model_version = $model_version,
        e.gnn_updated_at = datetime()
    """, rows=entity_records, model_version='graphsage_v1')

print('Embeddings upserted:', len(chunk_records), 'chunks,', len(entity_records), 'entities')


In [ ]:
# Demo retrieval: top-N entities для произвольного chunk
sample_chunk_idx = 0
sample_chunk_id = chunks_df.iloc[sample_chunk_idx]['chunk_id']
sample_text = chunks_df.iloc[sample_chunk_idx]['text'][:220]

with torch.no_grad():
    z = model(data.x_dict, data.edge_index_dict)
    scores = torch.matmul(z['entity'], z['chunk'][sample_chunk_idx])
    topk = torch.topk(scores, k=min(10, len(scores))).indices.cpu().tolist()

print('Chunk:', sample_chunk_id)
print('Text preview:', sample_text)
print('Top entities:')
for i, idx in enumerate(topk, 1):
    row = entities_df.iloc[idx]
    print(f'{i:02d}. {row.entity_id} | {row.entity_text}')


In [ ]:
driver.close()
